# NYU Deep Learning Midterm: Qwen 2B LoRA for Text-to-SVG

Fine-tunes `Qwen3.5-2B-Instruct` with LoRA on the competition's 50,000 training examples, then generates SVG submissions for the 1,000 test prompts.

**Competition constraints:**
- Valid XML with `<svg>` root element
- Maximum 16,000 characters
- Maximum 256 `<path>` elements
- Canvas: 256×256 pixels
- Allowed tags only: svg, g, path, rect, circle, ellipse, line, polyline, polygon, defs, use, symbol, clipPath, mask, linearGradient, radialGradient, stop, text, tspan, title, desc, style, pattern, marker, filter

In [ ]:
%pip install -q unsloth datasets trl transformers accelerate peft bitsandbytes pandas lxml

In [ ]:
import os
import re
import time
import random
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import torch
from datasets import Dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Kaggle competition data paths
DATA_DIR = "/kaggle/input/competitions/dl-spring-2026-svg-generation"
TRAIN_PATH = f"{DATA_DIR}/train.csv"
TEST_PATH = f"{DATA_DIR}/test.csv"
SUBMISSION_PATH = "/kaggle/working/submission.csv"

print(f"Torch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
CONFIG = {
    "model_name": "unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit",
    "max_seq_length": 2048,
    "lora_r": 16,
    "lora_alpha": 16,
    "learning_rate": 2e-4,
    "num_train_epochs": 1,
    "per_device_train_batch_size": 2,
    "gradient_accumulation_steps": 8,
    "warmup_ratio": 0.05,
    "weight_decay": 0.01,
    "logging_steps": 20,
    "eval_steps": 100,
    "save_steps": 200,
    "eval_size": 0.02,
    "max_new_tokens": 1024,
    "inference_batch_size": 8,
    "output_dir": "/kaggle/working/qwen2b_svg_lora",
}

CONFIG

In [ ]:
# SVG constraint constants (from competition rules)
MAX_SVG_CHARS = 16_000
MAX_PATH_ELEMENTS = 256
ALLOWED_TAGS = {
    "svg", "g", "path", "rect", "circle", "ellipse", "line",
    "polyline", "polygon", "defs", "use", "symbol", "clipPath",
    "mask", "linearGradient", "radialGradient", "stop", "text",
    "tspan", "title", "desc", "style", "pattern", "marker", "filter",
}


def _local_tag(element):
    """Strip XML namespace prefix from a tag, e.g. '{http://...}svg' -> 'svg'."""
    tag = element.tag
    return tag.split("}", 1)[-1] if "}" in tag else tag


def is_valid_svg(svg_text):
    """Return True if svg_text satisfies all competition constraints."""
    if not svg_text or not isinstance(svg_text, str):
        return False
    if len(svg_text) > MAX_SVG_CHARS:
        return False
    svg_text = svg_text.strip()
    if not svg_text.lower().startswith("<svg"):
        return False
    try:
        root = ET.fromstring(svg_text)
    except ET.ParseError:
        return False
    if _local_tag(root) != "svg":
        return False
    path_count = 0
    for el in root.iter():
        tag = _local_tag(el)
        if tag not in ALLOWED_TAGS:
            return False
        if tag == "path":
            path_count += 1
    if path_count > MAX_PATH_ELEMENTS:
        return False
    return True


def fallback_svg(_prompt=""):
    """Minimal valid 256x256 SVG used when generation fails."""
    return (
        '<svg xmlns="http://www.w3.org/2000/svg" '
        'width="256" height="256" viewBox="0 0 256 256">'
        '<rect x="0" y="0" width="256" height="256" fill="white"/>'
        '<circle cx="128" cy="128" r="64" fill="black"/>'
        '</svg>'
    )


# Quick sanity check
assert is_valid_svg(fallback_svg()), "Fallback SVG must be valid"
print("SVG validation helpers ready.")

In [ ]:
# Load the competition training data
train_df = pd.read_csv(TRAIN_PATH)
print(f"Raw training rows: {len(train_df)}")
print(train_df.dtypes)
print(train_df.head(2))

In [ ]:
# Filter training examples to only those with:
# - Non-empty prompt
# - SVG that satisfies all competition constraints
# (Trains the model to generate outputs that will actually score > 0)

train_df = train_df.dropna(subset=["prompt", "svg"])
train_df["prompt"] = train_df["prompt"].str.strip()
train_df["svg"] = train_df["svg"].str.strip()

# Filter: valid prompt
train_df = train_df[train_df["prompt"].str.len() > 0]

# Filter: SVG passes all competition constraints
print("Validating training SVGs (this may take a minute)...")
valid_mask = train_df["svg"].apply(is_valid_svg)
train_df = train_df[valid_mask].reset_index(drop=True)

print(f"Training rows after filtering: {len(train_df)}")
print(f"  Prompt length stats: {train_df['prompt'].str.len().describe().to_dict()}")
print(f"  SVG length stats:   {train_df['svg'].str.len().describe().to_dict()}")

In [ ]:
# Convert to HuggingFace Dataset and split into train / eval
full_ds = Dataset.from_pandas(train_df[["prompt", "svg"]], preserve_index=False)
full_ds = full_ds.shuffle(seed=SEED)

splits = full_ds.train_test_split(test_size=CONFIG["eval_size"], seed=SEED)
train_ds = splits["train"]
eval_ds = splits["test"]

print(f"Train rows: {len(train_ds)}")
print(f"Eval rows:  {len(eval_ds)}")
print("Sample:", train_ds[0]["prompt"][:80])

In [ ]:
SYSTEM_PROMPT = (
    "You generate compact, valid SVG markup from a natural language description. "
    "Output only SVG code. The root element must be <svg> with width=\"256\" height=\"256\" "
    "viewBox=\"0 0 256 256\". Use only allowed tags. Keep the output under 16000 characters."
)


def format_sft_text(example):
    text = (
        "<|im_start|>system\n"
        f"{SYSTEM_PROMPT}<|im_end|>\n"
        "<|im_start|>user\n"
        f"{example['prompt']}<|im_end|>\n"
        "<|im_start|>assistant\n"
        f"{example['svg']}<|im_end|>"
    )
    return {"text": text}


train_text = train_ds.map(format_sft_text, remove_columns=train_ds.column_names)
eval_text = eval_ds.map(format_sft_text, remove_columns=eval_ds.column_names)

print("Example training text (first 400 chars):")
print(train_text[0]["text"][:400])

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CONFIG["model_name"],
    max_seq_length=CONFIG["max_seq_length"],
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=CONFIG["lora_r"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=0,
    bias="none",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)

print(model.print_trainable_parameters())

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir=CONFIG["output_dir"],
    num_train_epochs=CONFIG["num_train_epochs"],
    per_device_train_batch_size=CONFIG["per_device_train_batch_size"],
    gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
    learning_rate=CONFIG["learning_rate"],
    warmup_ratio=CONFIG["warmup_ratio"],
    weight_decay=CONFIG["weight_decay"],
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=CONFIG["logging_steps"],
    eval_strategy="steps",
    eval_steps=CONFIG["eval_steps"],
    save_steps=CONFIG["save_steps"],
    save_total_limit=2,
    report_to="none",
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    seed=SEED,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_text,
    eval_dataset=eval_text,
    dataset_text_field="text",
    max_seq_length=CONFIG["max_seq_length"],
    packing=True,
    args=training_args,
)

train_result = trainer.train()
print(train_result)

In [ ]:
os.makedirs(CONFIG["output_dir"], exist_ok=True)
trainer.save_model(CONFIG["output_dir"])
tokenizer.save_pretrained(CONFIG["output_dir"])
print(f"Saved adapter + tokenizer to: {CONFIG['output_dir']}")

In [ ]:
FastLanguageModel.for_inference(model)
model.eval()

# Left-padding so all sequences in a batch end at the same position,
# allowing us to slice generated tokens uniformly with output[:, input_len:]
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

SVG_REGEX = re.compile(r"<svg.*?</svg>", flags=re.IGNORECASE | re.DOTALL)


def extract_svg(text):
    m = SVG_REGEX.search(text)
    return m.group(0).strip() if m else ""


def generate_svg_batch(prompts, max_new_tokens=None):
    if max_new_tokens is None:
        max_new_tokens = CONFIG["max_new_tokens"]

    chat_texts = [
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\n{p}<|im_end|>\n"
        f"<|im_start|>assistant\n"
        for p in prompts
    ]
    inputs = tokenizer(
        chat_texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=CONFIG["max_seq_length"] - max_new_tokens,
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.pad_token_id,
        )

    input_len = inputs["input_ids"].shape[1]
    results = []
    for output in output_ids:
        decoded = tokenizer.decode(output[input_len:], skip_special_tokens=True)
        svg = extract_svg(decoded)
        if not is_valid_svg(svg):
            svg = fallback_svg()
        results.append(svg)
    return results


# Smoke test
test_svgs = generate_svg_batch(["a simple blue circle on a white background"])
print(f"Valid: {is_valid_svg(test_svgs[0])}  Length: {len(test_svgs[0])}")
print(test_svgs[0][:300])

In [ ]:
test_df = pd.read_csv(TEST_PATH)
print(f"Test rows: {len(test_df)}")
print(test_df.head(2))

all_prompts = test_df["prompt"].tolist()
all_ids = test_df["id"].tolist()
batch_size = CONFIG["inference_batch_size"]

rows = []
invalid_count = 0
t0 = time.time()

for i in range(0, len(all_prompts), batch_size):
    batch_prompts = all_prompts[i:i + batch_size]
    batch_ids = all_ids[i:i + batch_size]
    svgs = generate_svg_batch(batch_prompts)
    for id_, svg in zip(batch_ids, svgs):
        if not is_valid_svg(svg):
            invalid_count += 1
        rows.append({"id": id_, "svg": svg})
    done = min(i + batch_size, len(all_prompts))
    if done % 100 == 0 or done == len(all_prompts):
        elapsed = (time.time() - t0) / 60
        print(f"  {done}/{len(all_prompts)} rows ({elapsed:.1f} min, {invalid_count} fallbacks)")

sub_df = pd.DataFrame(rows)
sub_df.to_csv(SUBMISSION_PATH, index=False)

elapsed_min = (time.time() - t0) / 60
print(f"\nSaved: {SUBMISSION_PATH}")
print(f"Rows: {len(sub_df)}")
print(f"Invalid/fallback count: {invalid_count} ({100*invalid_count/len(sub_df):.1f}%)")
print(f"Runtime (minutes): {elapsed_min:.2f}")
sub_df.head()